# Wind Turbine Foundation Analysis Params - REST API to JSON Schema

**Workspace ID:** 2677  
**Entity ID:** 12173  
**App URL:** https://demo.viktor.ai/workspaces/2677/app/editor/12173

This notebook reads the entity data through the VIKTOR REST API, converts the effective parameters to JSON Schema, executes the first useful method, and writes a Pydantic params model.


In [1]:
import os
import json
from pathlib import Path
from typing import Any

import requests
from dotenv import find_dotenv, load_dotenv

DOTENV_PATH = find_dotenv(usecwd=True)
load_dotenv(DOTENV_PATH, override=True)

print(f"Loaded .env: {DOTENV_PATH or '<not found>'}")


def get_env_int(*names: str, default: int) -> int:
    for name in names:
        raw_value = os.getenv(name)
        if raw_value and raw_value.strip():
            try:
                return int(raw_value)
            except ValueError as exc:
                raise ValueError(f"{name} must be an integer.") from exc
    return default



WORKSPACE_ID = get_env_int("WIND_TURBINE_FOUNDATION_ANALYSIS_WORKSPACE_ID", "VIKTOR_WORKSPACE_ID", default=2677)
ENTITY_ID = get_env_int("WIND_TURBINE_FOUNDATION_ANALYSIS_ENTITY_ID", "VIKTOR_ENTITY_ID", default=12173)
APP_OUTPUT_DIR = Path("wind_turbine_foundation_analysis")
OUTPUT_FILE = APP_OUTPUT_DIR / "input_schema.json"
METHODS_OUTPUT_FILE = APP_OUTPUT_DIR / "available_methods.json"
METHOD_RESULT_OUTPUT_FILE = APP_OUTPUT_DIR / "first_method_result.json"

print(f"Target: Workspace {WORKSPACE_ID}, Entity {ENTITY_ID}")

Loaded .env: /Users/alejandroduarte/Documents/nemetschek-agentic-demo/convert-app-params-to-schema/.env
Target: Workspace 2677, Entity 12173


## 1. Connect to VIKTOR REST API

The REST helper below follows the repository VIKTOR REST API rules: normalize the API base once, read the token from environment variables, do not print the token, centralize timeouts and errors, and pass query parameters through `params=...`.

In [2]:
# Initialize a small REST API client with a Personal Access Token.
# Supported token environment variables, in order:
# TOKEN_VK_APP, VIKTOR_TOKEN, VIKTOR_API_TOKEN.

TOKEN_ENV_NAMES = ("TOKEN_VK_APP", "VIKTOR_TOKEN", "VIKTOR_API_TOKEN")


def normalize_bearer_token(raw_token: str, *, env_var: str) -> str:
    token = raw_token.strip().strip('"').strip("'").strip()

    if token.startswith(f"{env_var}="):
        token = token.split("=", 1)[1].strip().strip('"').strip("'").strip()

    if token.lower().startswith("authorization:"):
        token = token.split(":", 1)[1].strip()

    if token.lower().startswith("bearer "):
        token = token.split(None, 1)[1].strip()

    token = token.strip().strip('"').strip("'").strip()
    if not token:
        raise ValueError(f"{env_var} is empty.")
    if any(ch.isspace() for ch in token):
        raise ValueError(f"{env_var} contains whitespace. Paste only the token value.")
    return token


def get_token() -> tuple[str, str]:
    for env_var in TOKEN_ENV_NAMES:
        raw_token = os.getenv(env_var)
        if raw_token and raw_token.strip():
            return normalize_bearer_token(raw_token, env_var=env_var), env_var
    raise ValueError(
        "Missing VIKTOR token. Set TOKEN_VK_APP or VIKTOR_TOKEN. "
        "VIKTOR_API_TOKEN is also accepted for backwards compatibility."
    )


def get_api_base() -> str:
    api_base = os.getenv("VIKTOR_API_BASE")
    if api_base and api_base.strip():
        base = api_base.strip().rstrip("/")
    else:
        environment = os.getenv("VIKTOR_ENVIRONMENT", "demo").strip().rstrip("/")
        if not environment:
            raise ValueError("Missing VIKTOR_ENVIRONMENT.")
        if environment.startswith("https://"):
            base = environment
        elif environment.startswith("http://"):
            base = "https://" + environment.removeprefix("http://")
        elif environment.endswith(".viktor.ai"):
            base = f"https://{environment}"
        else:
            base = f"https://{environment}.viktor.ai"

    base = base.rstrip("/")
    if base.startswith("http://"):
        base = "https://" + base.removeprefix("http://")
    if not base.startswith("https://"):
        raise ValueError("VIKTOR API base must resolve to an HTTPS URL.")
    return base if base.endswith("/api") else f"{base}/api"


class ViktorRestClient:
    def __init__(
        self,
        *,
        api_base: str,
        token: str,
        connect_timeout: float = 5.0,
        read_timeout: float = 30.0,
    ) -> None:
        token = token.strip()
        if not token:
            raise ValueError("Missing VIKTOR token.")

        base = api_base.strip().rstrip("/")
        self.api_base = base if base.endswith("/api") else f"{base}/api"
        self.timeout = (connect_timeout, read_timeout)
        self.session = requests.Session()
        self.session.headers.update(
            {
                "Authorization": f"Bearer {token}",
                "Accept": "application/json",
            }
        )

    def url(self, path_or_url: str) -> str:
        if path_or_url.startswith("http://") or path_or_url.startswith("https://"):
            return path_or_url
        return f"{self.api_base}/{path_or_url.lstrip('/')}"

    def check_response(self, response: requests.Response, *, action: str) -> None:
        if response.ok:
            return
        body = response.text[:500]
        raise RuntimeError(f"{action} failed (status={response.status_code}): {body}")

    def request_json(
        self,
        method: str,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
        json_body: dict[str, Any] | None = None,
        action: str = "VIKTOR REST request",
    ) -> dict[str, Any]:
        response = self.session.request(
            method=method.upper(),
            url=self.url(path_or_url),
            params=params,
            json=json_body,
            timeout=self.timeout,
        )
        self.check_response(response, action=action)
        if not response.content:
            return {}
        try:
            return response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"{action} did not return valid JSON: {response.text[:500]}"
            ) from exc

    def get_json(
        self,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
        action: str = "GET request",
    ) -> dict[str, Any]:
        return self.request_json("GET", path_or_url, params=params, action=action)


    def post_json(
        self,
        path_or_url: str,
        *,
        json_body: dict[str, Any] | None = None,
        action: str = "POST request",
    ) -> dict[str, Any]:
        return self.request_json("POST", path_or_url, json_body=json_body, action=action)


    def create_job(
        self,
        *,
        workspace_id: int,
        entity_id: int,
        method_name: str,
        params: dict[str, Any],
    ) -> dict[str, Any]:
        job = self.post_json(
            f"workspaces/{workspace_id}/entities/{entity_id}/jobs/",
            json_body={
                "method_name": method_name,
                "params": params,
                "poll_result": False,
            },
            action="Create VIKTOR job",
        )
        if job.get("url"):
            return self.poll_job(job["url"])
        if job.get("status") == "success":
            return job
        raise RuntimeError(f"Unexpected VIKTOR job response: {job}")

    def poll_job(self, job_url: str) -> dict[str, Any]:
        import time

        failed_statuses = {"failed", "cancelled", "error", "error_user", "error_timeout"}
        deadline = time.monotonic() + 300
        sleep_seconds = 0.8

        while time.monotonic() < deadline:
            job = self.get_json(job_url, action="Poll VIKTOR job")
            status = job.get("status")
            if status == "success":
                return job
            if status in failed_statuses:
                raise RuntimeError(f"VIKTOR job failed with status={status}: {job.get('error')}")
            time.sleep(sleep_seconds)
            sleep_seconds = min(sleep_seconds * 1.5, 5.0)

        raise TimeoutError("VIKTOR job did not finish within 300 seconds.")

    def compute_method(
        self,
        *,
        workspace_id: int,
        entity_id: int,
        method_name: str,
        params: dict[str, Any],
    ) -> dict[str, Any]:
        job = self.create_job(
            workspace_id=workspace_id,
            entity_id=entity_id,
            method_name=method_name,
            params=params,
        )
        result = job.get("result") or job.get("content")
        if not isinstance(result, dict):
            raise RuntimeError(f"VIKTOR job did not return a JSON object result: {job}")
        return result

    def build_entity_editor_url(self, *, workspace_id: int, entity_id: int) -> str:
        ui_base = self.api_base[:-4] if self.api_base.endswith("/api") else self.api_base
        return f"{ui_base}/workspaces/{workspace_id}/app/editor/{entity_id}"


api_token, token_env_var = get_token()
client = ViktorRestClient(api_base=get_api_base(), token=api_token)

print(f"Using VIKTOR token from environment variable: {token_env_var}")
print(f"REST API base URL: {client.api_base}")

workspaces = client.get_json(
    "workspaces/",
    params={"limit": 1, "offset": 0, "detail_level": "minimal"},
    action="List workspaces",
)
if "results" not in workspaces:
    raise RuntimeError(f"Unexpected workspace list response keys: {list(workspaces.keys())}")

print("Connected to VIKTOR REST API")
print(f"Workspace list response keys: {list(workspaces.keys())}")

Using VIKTOR token from environment variable: TOKEN_VK_APP
REST API base URL: https://demo.viktor.ai/api


Connected to VIKTOR REST API
Workspace list response keys: ['count', 'next', 'previous', 'results']


## 2. Fetch Entity and Parametrization

In [3]:
# Fetch the entity through REST.
entity = client.get_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/",
    params={
        "properties": "true",
        "clean_params": "true",
        "param_types": "true",
    },
    action="Get entity",
)

print(f"Entity: {entity.get('name', '<no name returned>')}")
print(f"Type: {entity.get('entity_type_name', entity.get('entity_type', '<no type returned>'))}")
print(f"Editor URL: {client.build_entity_editor_url(workspace_id=WORKSPACE_ID, entity_id=ENTITY_ID)}")
print(f"Response keys: {list(entity.keys())}")


def extract_saved_params(entity_payload: dict[str, Any]) -> dict[str, Any]:
    """Extract saved params/properties from a VIKTOR REST entity response."""
    for key in ("properties", "params", "last_saved_params"):
        value = entity_payload.get(key)
        if isinstance(value, dict):
            return value

    for key in ("last_saved_revision", "latest_revision", "revision"):
        value = entity_payload.get(key)
        if isinstance(value, dict) and isinstance(value.get("params"), dict):
            return value["params"]

    raise KeyError(
        "No saved params/properties found in the REST response. "
        f"Available keys: {list(entity_payload.keys())}"
    )


params = extract_saved_params(entity)
print(f"\nParameter keys: {list(params.keys()) if params else 'None'}")

Entity: Foundation 1
Type: Controller
Editor URL: https://demo.viktor.ai/workspaces/2677/app/editor/12173
Response keys: ['id', 'name', 'properties', 'summary_status', 'summary', 'summary_updated_at', 'deleted', 'locked', 'locked_by', 'entity_type', 'entity_type_name', 'updated_at', 'parent_count', 'actions', 'param_types', 'created_by', 'created_at', 'last_updated_by', 'path', 'show_on_dashboard']

Parameter keys: ['step_geo', 'step_geo_tech']


## 3. Get Available Methods

The entity type exposes view controller methods. The editor parametrization payload can also expose button/action methods.

In [4]:
def set_nested_default(target: dict[str, Any], dotted_path: str, value: Any) -> None:
    keys = [part for part in dotted_path.split(".") if part]
    if not keys:
        return
    current = target
    for key in keys[:-1]:
        child = current.setdefault(key, {})
        if not isinstance(child, dict):
            child = {}
            current[key] = child
        current = child
    current[keys[-1]] = value


def collect_declared_defaults(nodes: Any) -> dict[str, Any]:
    defaults: dict[str, Any] = {}

    def walk(value: Any) -> None:
        if isinstance(value, list):
            for item in value:
                walk(item)
            return
        if not isinstance(value, dict):
            return

        path = value.get("name") or value.get("parametrization_path")
        if path and "default" in value:
            set_nested_default(defaults, path, value["default"])

        for child_key in ("content", "children", "items"):
            walk(value.get(child_key))

    walk(nodes)
    return defaults


def deep_merge(defaults: Any, overrides: Any) -> Any:
    if isinstance(defaults, dict) and isinstance(overrides, dict):
        merged = dict(defaults)
        for key, value in overrides.items():
            merged[key] = deep_merge(merged.get(key), value)
        return merged
    if overrides is not None:
        return overrides
    return defaults


def collect_view_methods(entity_type_payload: dict[str, Any]) -> list[dict[str, Any]]:
    methods: list[dict[str, Any]] = []
    for view in entity_type_payload.get("views") or []:
        if not isinstance(view, dict):
            continue
        method = view.get("controller_method") or view.get("method_name")
        if method:
            methods.append(
                {
                    "method_name": method,
                    "source": "entity_type.views",
                    "label": view.get("label"),
                    "view_type": view.get("view_type"),
                    "automatic_update": view.get("automatic_update"),
                }
            )
    return methods


def collect_parametrization_methods(nodes: Any) -> list[dict[str, Any]]:
    methods: list[dict[str, Any]] = []

    def walk(value: Any) -> None:
        if isinstance(value, list):
            for item in value:
                walk(item)
            return
        if not isinstance(value, dict):
            return

        method = value.get("method")
        if method:
            methods.append(
                {
                    "method_name": method,
                    "source": "parametrization",
                    "label": value.get("ui_name") or value.get("title"),
                    "node_type": value.get("type"),
                    "path": value.get("name") or value.get("parametrization_path"),
                }
            )

        for child_key in ("content", "children", "items"):
            walk(value.get(child_key))

    walk(nodes)
    return methods


def deduplicate_methods(methods: list[dict[str, Any]]) -> list[dict[str, Any]]:
    deduped: dict[str, dict[str, Any]] = {}
    for item in methods:
        deduped[item["method_name"]] = item
    return list(deduped.values())


entity_type = client.get_json(
    f"workspaces/{WORKSPACE_ID}/entity_types/{entity['entity_type']}/",
    action="Get entity type",
)

editor_session = client.post_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/session/",
    action="Create editor session",
)

parametrization = client.post_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/parametrization/",
    json_body={
        "editor_session": editor_session["editor_session"],
        "params": {},
    },
    action="Get parametrization",
)

parametrization_nodes = (parametrization.get("content") or {}).get("parametrization", [])
declared_defaults = collect_declared_defaults(parametrization_nodes)
effective_params = deep_merge(declared_defaults, params)
params = effective_params

available_methods = deduplicate_methods(
    collect_view_methods(entity_type) + collect_parametrization_methods(parametrization_nodes)
)

methods_output_path = Path(METHODS_OUTPUT_FILE)
methods_output_path.parent.mkdir(parents=True, exist_ok=True)
with open(methods_output_path, "w", encoding="utf-8") as f:
    json.dump(available_methods, f, indent=2)

print(f"Declared default top-level keys: {list(declared_defaults.keys())}")
print(f"Effective parameter keys: {list(params.keys())}")
print(f"Available methods: {len(available_methods)}")
print(json.dumps(available_methods, indent=2, default=str))
print(f"Methods saved to: {methods_output_path.absolute()}")

Declared default top-level keys: ['step_geo', 'step_geo_tech']
Effective parameter keys: ['step_geo', 'step_geo_tech']
Available methods: 7
[
  {
    "method_name": "view_geometry",
    "source": "entity_type.views",
    "label": "3D Geometry",
    "view_type": "geometry",
    "automatic_update": true
  },
  {
    "method_name": "view_results",
    "source": "entity_type.views",
    "label": "Results Summary",
    "view_type": "data",
    "automatic_update": false
  },
  {
    "method_name": "view_pile_reactions",
    "source": "entity_type.views",
    "label": "Pile Reactions",
    "view_type": "table",
    "automatic_update": false
  },
  {
    "method_name": "view_2d_internal_forces",
    "source": "entity_type.views",
    "label": "2D Internal Forces",
    "view_type": "table",
    "automatic_update": false
  },
  {
    "method_name": "view_mxd_plus_plot",
    "source": "entity_type.views",
    "label": "2D Moment Contour Plots",
    "view_type": "plotly",
    "automatic_update": f

## 4. Execute First Available View Method

The notebook calls the first available DataView/TableView-style method with the effective params and saves the selected result payload. It uses `POST /api/workspaces/{workspace_id}/entities/{entity_id}/jobs/` with `method_name`, `params`, and `poll_result: false`, then polls the returned `url` until `status == "success"`.

Result payload keys can include `data`, `table`, `download`, `geometry`, `plotly`, `geojson`, `web`, `pdf`, `image`, `ifc`, `optimization`, and `set_params`.

In [5]:
RESULT_KEY_PRIORITY = [
    "data",
    "table",
    "download",
    "geometry",
    "plotly",
    "geojson",
    "web",
    "pdf",
    "image",
    "ifc",
    "optimization",
    "set_params",
]

# VIKTOR method result payload keys can include:
# data, table, download, geometry, plotly, geojson, web, pdf, image, ifc, optimization, set_params.


def rank_methods_for_output(methods: list[dict[str, Any]]) -> list[dict[str, Any]]:
    ranked: list[dict[str, Any]] = []

    def add_matching(predicate) -> None:
        for method in methods:
            if method not in ranked and predicate(method):
                ranked.append(method)

    add_matching(lambda method: method.get("view_type") in {"data", "table"})
    add_matching(lambda method: bool(method.get("automatic_update")))
    add_matching(lambda method: method.get("source") == "entity_type.views" or bool(method.get("view_type")))
    add_matching(lambda method: True)
    return ranked


def select_first_result_payload(result: dict[str, Any]) -> tuple[str, Any]:
    for key in RESULT_KEY_PRIORITY:
        if key in result:
            return key, result[key]
    return "result", result


attempted_methods: list[dict[str, Any]] = []
method_result_output: dict[str, Any] | None = None

for selected_method in rank_methods_for_output(available_methods):
    try:
        method_result = client.compute_method(
            workspace_id=WORKSPACE_ID,
            entity_id=ENTITY_ID,
            method_name=selected_method["method_name"],
            params=params,
        )
    except Exception as exc:
        attempted_methods.append(
            {
                "method": selected_method,
                "status": "failed",
                "error": str(exc)[:500],
            }
        )
        print(f"Method failed: {selected_method['method_name']} ({exc})")
        continue

    result_key, selected_result_payload = select_first_result_payload(method_result)
    attempted_methods.append(
        {
            "method": selected_method,
            "status": "success",
            "result_key": result_key,
        }
    )
    method_result_output = {
        "method": selected_method,
        "result_key": result_key,
        "result": selected_result_payload,
        "attempted_methods": attempted_methods,
    }
    break

if method_result_output is None:
    method_result_output = {
        "status": "method_execution_failed",
        "message": "No available VIKTOR method executed successfully with the effective parameters.",
        "attempted_methods": attempted_methods,
    }

method_result_output_path = Path(METHOD_RESULT_OUTPUT_FILE)
method_result_output_path.parent.mkdir(parents=True, exist_ok=True)
with open(method_result_output_path, "w", encoding="utf-8") as f:
    json.dump(method_result_output, f, indent=2)

if "method" in method_result_output:
    print(f"Executed method: {method_result_output['method']['method_name']}")
    print(f"Selected result key: {method_result_output['result_key']}")
else:
    print(method_result_output["message"])
print(f"Method result saved to: {method_result_output_path.absolute()}")


Method failed: view_results (VIKTOR job failed with status=failed: {'type': 'error_code', 'message': '', 'invalid_fields': {}})


Method failed: view_pile_reactions (VIKTOR job failed with status=failed: {'type': 'error_code', 'message': '', 'invalid_fields': {}})


Method failed: view_2d_internal_forces (VIKTOR job failed with status=failed: {'type': 'error_code', 'message': '', 'invalid_fields': {}})


Executed method: view_geometry
Selected result key: geometry
Method result saved to: /Users/alejandroduarte/Documents/nemetschek-agentic-demo/convert-app-params-to-schema/wind_turbine_foundation_analysis/first_method_result.json


## 5. Infer JSON Schema From Effective Params

The schema is inferred from declared defaults merged with saved entity properties. Saved properties win over declared defaults.

In [6]:
def infer_json_schema_from_params(
    params: dict[str, Any],
    *,
    include_defaults: bool = True,
) -> dict[str, Any]:
    schema: dict[str, Any] = {
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    }
    required: list[str] = []

    for key, value in params.items():
        property_schema = infer_type_from_value(value, include_defaults=include_defaults)
        if include_defaults:
            property_schema["default"] = value
        else:
            required.append(key)
        schema["properties"][key] = property_schema

    if required:
        schema["required"] = required
    return schema


def infer_type_from_value(value: Any, *, include_defaults: bool = True) -> dict[str, Any]:
    if isinstance(value, bool):
        return {"type": "boolean"}
    if isinstance(value, int) and not isinstance(value, bool):
        return {"type": "integer"}
    if isinstance(value, float):
        return {"type": "number"}
    if isinstance(value, str):
        return {"type": "string"}
    if isinstance(value, list):
        return {
            "type": "array",
            "items": infer_array_item_schema(value),
        }
    if isinstance(value, dict):
        return infer_json_schema_from_params(value, include_defaults=include_defaults)
    if value is None:
        return {"type": ["string", "null"]}
    return {"type": "string"}


def infer_array_item_schema(values: list[Any]) -> dict[str, Any]:
    if not values:
        return {"type": "string"}
    if all(isinstance(item, dict) for item in values):
        merged_shape: dict[str, Any] = {}
        for item in values:
            merged_shape = deep_merge(merged_shape, item)
        return infer_json_schema_from_params(merged_shape, include_defaults=False)
    return infer_type_from_value(values[0], include_defaults=False)


if params:
    schema = infer_json_schema_from_params(params)
    print("Schema generated with defaults")
    print(f"\nProperties: {list(schema['properties'].keys())}")
else:
    print("No parameters found; creating an empty schema")
    schema = {
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    }

Schema generated with defaults

Properties: ['step_geo', 'step_geo_tech']


## 6. Preview the Schema

In [7]:
print(json.dumps(schema, indent=2))

{
  "type": "object",
  "properties": {
    "step_geo": {
      "type": "object",
      "properties": {
        "sec_mast": {
          "type": "object",
          "properties": {
            "mast_diameter": {
              "type": "number",
              "default": 5.0
            },
            "mast_vertical_load": {
              "type": "number",
              "default": 4000.0
            },
            "mast_horizontal_load": {
              "type": "number",
              "default": 1500.0
            },
            "mast_moment": {
              "type": "number",
              "default": 150000.0
            }
          },
          "additionalProperties": false,
          "default": {
            "mast_diameter": 5.0,
            "mast_vertical_load": 4000.0,
            "mast_horizontal_load": 1500.0,
            "mast_moment": 150000.0
          }
        },
        "sec_plate": {
          "type": "object",
          "properties": {
            "slab_diameter": {
        

## 7. Save Schema to JSON File

In [8]:
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

print(f"Schema saved to: {output_path.absolute()}")

Schema saved to: /Users/alejandroduarte/Documents/nemetschek-agentic-demo/convert-app-params-to-schema/wind_turbine_foundation_analysis/input_schema.json


## 8. Display Effective Parameters

These are the declared defaults after applying saved entity properties on top.

In [9]:
print("Effective parameters used for schema defaults:")
print(json.dumps(params, indent=2, default=str))

Effective parameters used for schema defaults:
{
  "step_geo": {
    "sec_mast": {
      "mast_diameter": 5.0,
      "mast_vertical_load": 4000.0,
      "mast_horizontal_load": 1500.0,
      "mast_moment": 150000.0
    },
    "sec_plate": {
      "slab_diameter": 20.0,
      "slab_thickness": 4.5,
      "plate_edge_thickness": 1.0,
      "pedestal_height": 1.0
    },
    "sec_piles": {
      "num_piles": 30,
      "pile_length": 20.0,
      "pile_diameter": 500,
      "pile_edge_distance": 600
    }
  },
  "step_geo_tech": {
    "sec_tip": {
      "tip_stiffness": 50000.0
    },
    "sec_lateral": {
      "lateral_stiffness": 10000.0
    }
  }
}


## 9. Generate Pydantic Param Models

The generated `params_model.py` file mirrors `input_schema.json` with explicit Pydantic fields and defaults, so `{}` remains valid when the VIKTOR app has defaults.


In [10]:
from __future__ import annotations

import keyword
import re
from copy import deepcopy
from pathlib import Path
from typing import Any


def to_pascal_case(value: str) -> str:
    words = re.findall(r"[A-Za-z0-9]+", value)
    return "".join(word[:1].upper() + word[1:] for word in words) or "Generated"


def to_snake_case(value: str) -> str:
    value = re.sub(r"[^A-Za-z0-9]+", "_", value).strip("_")
    value = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", value).lower() or "field"
    if value[0].isdigit():
        value = f"field_{value}"
    if keyword.iskeyword(value):
        value = f"{value}_"
    return value


def class_name_for(parts: list[str]) -> str:
    return "".join(to_pascal_case(part) for part in parts) or "Params"


def python_literal(value: Any) -> str:
    return repr(value)


def is_object_schema(field_schema: dict[str, Any]) -> bool:
    return field_schema.get("type") == "object" or "properties" in field_schema


def is_array_schema(field_schema: dict[str, Any]) -> bool:
    return field_schema.get("type") == "array"


def item_annotation_from_list(annotation: str) -> str | None:
    if annotation.startswith("list[") and annotation.endswith("]"):
        return annotation[5:-1]
    return None


def field_default_expr(field_schema: dict[str, Any], annotation: str) -> str:
    if is_object_schema(field_schema):
        if "default" in field_schema:
            return f"default_factory={annotation}"
        return "..."

    if is_array_schema(field_schema) and "default" in field_schema:
        default = field_schema["default"]
        item_annotation = item_annotation_from_list(annotation)
        item_schema = field_schema.get("items") or {}
        if item_annotation and is_object_schema(item_schema):
            return f"default_factory=lambda: [{item_annotation}.model_validate(item) for item in deepcopy({python_literal(default)})]"
        return f"default_factory=lambda: deepcopy({python_literal(default)})"

    if "default" not in field_schema:
        return "..."

    default = field_schema["default"]
    if isinstance(default, list):
        return f"default_factory=lambda: deepcopy({python_literal(default)})"
    return f"default={python_literal(default)}"


def annotation_for_schema(
    field_schema: dict[str, Any],
    *,
    path: list[str],
    classes: dict[str, list[str]],
) -> str:
    schema_type = field_schema.get("type")
    if isinstance(schema_type, list):
        non_null = [item for item in schema_type if item != "null"]
        if len(non_null) == 1:
            return f"{annotation_for_schema({**field_schema, 'type': non_null[0]}, path=path, classes=classes)} | None"
        return "Any"
    if is_object_schema(field_schema):
        name = class_name_for(path)
        add_class(name, field_schema, path=path, classes=classes)
        return name
    if schema_type == "array":
        item_schema = field_schema.get("items") or {"type": "string"}
        item_annotation = annotation_for_schema(item_schema, path=path + ["item"], classes=classes)
        return f"list[{item_annotation}]"
    if schema_type == "integer":
        return "int"
    if schema_type == "number":
        return "float"
    if schema_type == "boolean":
        return "bool"
    if schema_type == "string":
        return "str"
    return "Any"


def add_class(
    name: str,
    object_schema: dict[str, Any],
    *,
    path: list[str],
    classes: dict[str, list[str]],
) -> None:
    if name in classes:
        return

    lines = [f"class {name}(BaseModel):"]
    properties = object_schema.get("properties") or {}
    if not properties:
        lines.append("    pass")
        classes[name] = lines
        return

    for raw_field_name, field_schema in properties.items():
        field_name = to_snake_case(raw_field_name)
        annotation = annotation_for_schema(field_schema, path=path + [raw_field_name], classes=classes)
        default_expr = field_default_expr(field_schema, annotation)
        alias_expr = f", alias={python_literal(raw_field_name)}" if field_name != raw_field_name else ""
        lines.append(f"    {field_name}: {annotation} = Field({default_expr}{alias_expr})")

    classes[name] = lines


root_model_name = class_name_for([APP_OUTPUT_DIR.name, "params"])
classes: dict[str, list[str]] = {}
add_class(root_model_name, schema, path=[APP_OUTPUT_DIR.name, "params"], classes=classes)

ordered_class_names = [name for name in classes if name != root_model_name] + [root_model_name]
model_source = "\n".join(
    [
        "from __future__ import annotations",
        "",
        "from copy import deepcopy",
        "from typing import Any",
        "",
        "from pydantic import BaseModel, Field",
        "",
        "",
    ]
)
model_source += "\n\n\n".join("\n".join(classes[name]) for name in ordered_class_names)
model_source += "\n"

model_path = Path(APP_OUTPUT_DIR) / "params_model.py"
model_path.parent.mkdir(parents=True, exist_ok=True)
model_path.write_text(model_source, encoding="utf-8")

print(f"Pydantic model saved to: {model_path.absolute()}")
print(model_source)


Pydantic model saved to: /Users/alejandroduarte/Documents/nemetschek-agentic-demo/convert-app-params-to-schema/wind_turbine_foundation_analysis/params_model.py
from __future__ import annotations

from copy import deepcopy
from typing import Any

from pydantic import BaseModel, Field

class WindTurbineFoundationAnalysisParamsStepGeoSecMast(BaseModel):
    mast_diameter: float = Field(default=5.0)
    mast_vertical_load: float = Field(default=4000.0)
    mast_horizontal_load: float = Field(default=1500.0)
    mast_moment: float = Field(default=150000.0)


class WindTurbineFoundationAnalysisParamsStepGeoSecPlate(BaseModel):
    slab_diameter: float = Field(default=20.0)
    slab_thickness: float = Field(default=4.5)
    plate_edge_thickness: float = Field(default=1.0)
    pedestal_height: float = Field(default=1.0)


class WindTurbineFoundationAnalysisParamsStepGeoSecPiles(BaseModel):
    num_piles: int = Field(default=30)
    pile_length: float = Field(default=20.0)
    pile_diameter: in